# M1 · 01 — Champion recovery (CPU)

**Purpose.** Identify the strongest **valid, deployable** existing W2 configuration from the frozen boost-A grid with the predeclared p6g selection rule. No new model, no grid recomputation.

| | |
|---|---|
| Runtime | CPU, High-RAM recommended (unit crops of the full grid) |
| Prerequisites | notebook 00 status COMPLETE |
| Inputs | `p6_fast_track/boost/benchmark/grid.csv`, `stack/maps/*.npy`, `stack/candidate.json`, `recovery/export_parity/*`, B0 / B1 OOF maps, M9 baseline selection |
| Outputs | `selection/{champion_candidates.csv, champion_selection.json, champion_recipe.yaml, champion_provenance.json, blocked_candidates.csv, candidate_per_zone.csv, protocol_consistency.csv}`, `reports/recovery_report.md` |
| Next | `02_OFFICIAL_VALIDATION.ipynb` when the selection status is COMPLETE |

**Selection rule (predeclared, `configs/p6/full_gpu.yaml` → `selection`).**
* For each outer zone, ONE cell is chosen on the other zones and applied to that zone in both tasks.
* The objective is `disc`.
* Eligibility requires a worst zone ≥ 0.153414 and the relative geographic rule.
* A replacement must gain ≥ 0.005 and be no worse on ≥ 2 of 3 zones.

The 0.1758 geographic floor is reported per candidate (`geo_floor_met`). It blocks only if you set `M1_CONFIG = {'selection': {'geo_floor_blocking': True}}`.

## 1 · Configuration

In [ ]:
CONFIG = {'repo': 'mybenkhadda/GEMS---drivendata', 'ref': 'main', 'drive_root': '/content/drive/MyDrive/GEMS_2',
          'local_root': '/content/gems_work', 'reuse_cached': True, 'strict_provenance': True,
          'overwrite_existing': False, 'device': 'cpu'}
PRIVATE_REPO = False
M1_CONFIG = {}
EXPECT = {'code_rev': 'm1.1-r1', 'commit': None}

## 2 · Drive, strict clone, version lock, shared session

`M1Session.attach` stages cached data (never rebuilt), builds M9 → P6 → p6g in `mode: recovery` with its output root = the M1 root, and refuses to start without a COMPLETE notebook-00 inventory.

In [ ]:
import base64, os, subprocess, sys, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
def git(*a, check=True):
    env = None
    if PRIVATE_REPO:
        from google.colab import userdata
        hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
        env = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}
    r = subprocess.run(['git', *a], capture_output=True, text=True, env=env)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(a[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r
if not Path(CODE, '.git').exists():
    git('clone', '-q', 'https://github.com/' + CONFIG['repo'] + '.git', CODE)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin')
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + CONFIG['ref'], check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + CONFIG['ref'])
else:
    git('-C', CODE, 'checkout', '-q', '--detach', CONFIG['ref'])
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'] and (not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']))
from gems_bench.leaderboard_focus.session import M1Session
m1 = M1Session.attach(CONFIG, M1_CONFIG)
print('HEAD', HEAD, '| M1 root', m1.ws.root, '| staged W2 inputs', m1.staged)

## 3 · Recover the champion (reloads while the fingerprint is unchanged)

In [ ]:
from IPython.display import Markdown, display
from gems_bench.leaderboard_focus.champion import recover
M = recover(m1)
SEL = m1.ws.read_json('selection/champion_selection.json') or {}
print('SELECTION STATUS:', M['status'], '| champion:', SEL.get('champion'), '| decision:', SEL.get('decision'), '| recipe:', SEL.get('recipe_id'))
for r in M['reasons']:
    print('  reason:', r)
C = m1.ws.load_table('selection/champion_candidates.csv')
display(C[['candidate_id', 'role', 'status', 'eligible', 'disc_dti', 'disc_worst', 'geo_dti', 'geo_floor_met', 'recorded_disc_dti', 'decision']])

## 4 · Verification checklist

1. `SELECTION STATUS: COMPLETE` with a champion and a recipe id.
2. `selection/protocol_consistency.csv`: every `ok` True.
3. `blocked_candidates.csv`: every blocked candidate has a reason (missing maps, worst-zone floor, historical role, …).
4. `recorded_*` columns are historical records; `disc_dti` / `geo_dti` are the recomputed nested single-policy numbers. They differ by construction when the record came from separate disc / geo selections.

## 5 · Troubleshooting

* `StageBlocked: artifact_inventory.json ...`: run notebook 00.
* `NO_ELIGIBLE_CANDIDATE`: no candidate passes the predeclared rule. M1 stops (BLOCKED) instead of exporting the reference silently; inspect `eligibility_reasons`.
* Memory error: switch to a High-RAM runtime (each unit crop of the B0 / B1 / W2 maps is held in RAM once).

## 6 · Next

`02_OFFICIAL_VALIDATION.ipynb`.

In [ ]:
display(Markdown(m1.ws.path('reports/recovery_report.md').read_text(encoding='utf-8')))